# Chemical Compatibility, Performance, and Root Cause Analysis

End-to-end demonstration of NeqSim's production-chemistry toolkit:

1. Build a chemical inventory and screen for compatibility
2. Predict scale (CaCO₃, BaSO₄) and evaluate inhibitor performance
3. Size an acid stimulation job for carbonate scale removal
4. Size a triazine H₂S scavenger system
5. Run root-cause analysis on a multi-symptom incident

All classes live in `neqsim.process.chemistry.*` and follow the standard NeqSim
pattern: setters → `evaluate()` / `analyse()` → getters + `toMap()`/`toJson()`.

## Setup

Use `neqsim_dev_setup` so the latest workspace classes are loaded from `target/classes`.

In [ ]:
import os
import sys
from pathlib import Path

def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for c in candidates:
        if (c / "pom.xml").exists() and (c / "devtools" / "neqsim_dev_setup.py").exists():
            return c
    raise RuntimeError("Could not find NeqSim project root. Set NEQSIM_PROJECT_ROOT.")

PROJECT_ROOT = find_neqsim_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))
from neqsim_dev_setup import neqsim_init, neqsim_classes
ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=True)
ns = neqsim_classes(ns)

In [ ]:
ProductionChemical = ns.JClass("neqsim.process.chemistry.ProductionChemical")
ChemicalCompatibilityAssessor = ns.JClass(
    "neqsim.process.chemistry.ChemicalCompatibilityAssessor")
ScaleInhibitorPerformance = ns.JClass(
    "neqsim.process.chemistry.scale.ScaleInhibitorPerformance")
ScaleControlAssessor = ns.JClass(
    "neqsim.process.chemistry.scale.ScaleControlAssessor")
CorrosionInhibitorPerformance = ns.JClass(
    "neqsim.process.chemistry.corrosion.CorrosionInhibitorPerformance")
AcidTreatmentSimulator = ns.JClass(
    "neqsim.process.chemistry.acid.AcidTreatmentSimulator")
H2SScavengerPerformance = ns.JClass(
    "neqsim.process.chemistry.scavenger.H2SScavengerPerformance")
ScalePredictionCalculator = ns.JClass(
    "neqsim.pvtsimulation.flowassurance.ScalePredictionCalculator")
RootCauseAnalyser = ns.JClass(
    "neqsim.process.chemistry.rca.RootCauseAnalyser")
Symptom = ns.JClass("neqsim.process.chemistry.rca.Symptom")

## 1. Chemical Compatibility Screening

Imagine a topside injection manifold where corrosion inhibitor (cationic
quaternary), scale inhibitor (anionic phosphonate), and an MEA-triazine H₂S
scavenger meet at 85 °C with 2500 mg/L Ca²⁺ in the produced water.

In [ ]:
ci = ProductionChemical.corrosionInhibitor("CI-quat-50", 50.0)
si = ProductionChemical.scaleInhibitor("SI-phosphonate-15", 15.0)
scav = ProductionChemical.h2sScavenger("MEA-triazine-100", 100.0)

compat = ChemicalCompatibilityAssessor()
compat.addChemical(ci)
compat.addChemical(si)
compat.addChemical(scav)
compat.setTemperatureCelsius(85.0)
compat.setCalciumMgL(2500.0)
compat.setMaterial("carbon_steel")
compat.evaluate()

print("Verdict:", compat.getVerdict())
issues = list(compat.getIssues())
for issue in issues:
    print("  -", issue)

## 2. Scale Prediction + Inhibitor Performance

In [ ]:
predictor = ScalePredictionCalculator()
predictor.setTemperatureCelsius(80.0)
predictor.setPH(7.0)
predictor.setCalciumConcentration(2500.0)
predictor.setBicarbonateConcentration(800.0)
predictor.setSulphateConcentration(500.0)
predictor.setBariumConcentration(50.0)
predictor.setSodiumConcentration(20000.0)
predictor.setTotalDissolvedSolids(80000.0)
predictor.calculate()

si_caco3 = predictor.getCaCO3SaturationIndex()
si_baso4 = predictor.getBaSO4SaturationIndex()
print(f"Uninhibited SI(CaCO3) = {si_caco3:.2f}")
print(f"Uninhibited SI(BaSO4) = {si_baso4:.2f}")

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

doses = np.linspace(2.0, 50.0, 25)
residuals = []
for dose in doses:
    sip = ScaleInhibitorPerformance()
    sip.setScaleType(ScaleInhibitorPerformance.ScaleType.CACO3)
    sip.setInhibitorChemistry(
        ScaleInhibitorPerformance.InhibitorChemistry.PHOSPHONATE)
    sip.setTemperatureCelsius(80.0)
    sip.setCalciumMgL(2500.0)
    sip.setSaturationRatio(10.0 ** max(0.0, si_caco3))
    sip.setTdsMgL(80000.0)
    sip.setAvailableDoseMgL(float(dose))
    sip.evaluate()

    assessor = ScaleControlAssessor(predictor)
    assessor.addInhibitor(ScaleInhibitorPerformance.ScaleType.CACO3, sip)
    assessor.evaluate()
    residuals.append(assessor.getResidualSI(
        ScaleInhibitorPerformance.ScaleType.CACO3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(doses, residuals, "b-o", lw=2, markersize=4)
ax.axhline(0.5, color="r", linestyle="--", label="Scaling threshold (SI=0.5)")
ax.set_xlabel("Inhibitor dose [mg/L]")
ax.set_ylabel("Residual SI(CaCO3) [-]")
ax.set_title("Scale control vs. phosphonate inhibitor dose")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("chemical_compatibility_dose_response.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Acid Treatment Sizing

Size an HCl matrix-acid job to remove 500 kg of CaCO₃ scale at 80 °C.

In [ ]:
acid = AcidTreatmentSimulator()
acid.setAcidType(AcidTreatmentSimulator.AcidType.HCL)
acid.setAcidVolumeM3(1.0)
acid.setAcidWtPct(15.0)
acid.setScaleMassKg(500.0)
acid.setScaleMineralogy("CACO3")
acid.setTemperatureCelsius(80.0)
acid.setMaterial("carbon_steel")
acid.evaluate()

print(f"Scale dissolvable: {acid.getDissolvableScaleKg():.1f} kg")
print(f"Scale dissolved: {acid.getScaleDissolvedKg():.1f} kg")
print(f"Dissolution fraction: {acid.getDissolutionFractionPct():.1f} %")
print(f"CO2 generated: {acid.getCO2GeneratedKg():.1f} kg")
print(f"Spent acid pH: {acid.getSpentAcidPH():.2f}")
print("Warnings:", list(acid.getWarnings()))

## 4. H₂S Scavenger Sizing

Size an MEA-triazine scavenger system for 2 MMSm³/d gas at 150 ppm H₂S inlet,
4 ppm sales-gas target.

In [ ]:
scav_perf = H2SScavengerPerformance()
scav_perf.setScavengerChemistry(
    H2SScavengerPerformance.ScavengerChemistry.MEA_TRIAZINE)
scav_perf.setGasFlowSm3PerDay(2.0e6)
scav_perf.setH2SInletPpm(150.0)
scav_perf.setH2SOutletTargetPpm(4.0)
scav_perf.setTemperatureCelsius(50.0)
scav_perf.setInventoryKg(5000.0)
scav_perf.evaluate()

print(f"H2S removal demand: {scav_perf.getH2SRemovedKgPerDay():.1f} kg/d")
print(f"Required dose: {scav_perf.getRequiredDoseLPerDay():.0f} L/d")
print(f"Breakthrough time: {scav_perf.getBreakthroughDays():.1f} d")
print("Warnings:", list(scav_perf.getWarnings()))

## 5. Root Cause Analysis

An incident: deposit found in the cooler tubes, wall thinning observed in the
downstream flowline, and pH excursion. Run the analyser with full process
context plus the compatibility assessor.

In [ ]:
rca = RootCauseAnalyser()
rca.setTemperatureCelsius(75.0)
rca.setPH(5.5)
rca.setCalciumMgL(2500.0)
rca.setCO2PartialPressureBar(2.0)
rca.setOxygenPpb(80.0)
rca.setMaterial("carbon_steel")
rca.setCompatibilityAssessor(compat)

rca.addSymptom(Symptom(Symptom.Category.DEPOSIT,
    "White crystalline scale found in cooler tubes")
    .withMeasurement("depositMassGrams", 250.0))
rca.addSymptom(Symptom(Symptom.Category.CORROSION,
    "Internal wall thinning in downstream flowline")
    .withMeasurement("corrosionRateMmYr", 1.2))
rca.addSymptom(Symptom(Symptom.Category.PH_EXCURSION,
    "Outlet pH dropped to 5.5"))
rca.analyse()

primary = rca.getPrimary()
print("PRIMARY:", primary.getCode(), "score=%.2f" % primary.getScore())
print("  evidence:", primary.getEvidence())
print("  recommendation:", primary.getRecommendation())
print()
for c in list(rca.getCandidates())[1:]:
    print(f"{c.getTag()}: {c.getCode()}  score={c.getScore():.2f}")
    print("  evidence:", c.getEvidence())

In [ ]:
candidates = list(rca.getCandidates())
codes = [str(c.getCode()) for c in candidates]
scores = [float(c.getScore()) for c in candidates]
tags = [str(c.getTag()) for c in candidates]
colours = {"PRIMARY": "#d62728", "CONTRIBUTING": "#ff7f0e",
           "POSSIBLE": "#1f77b4", "RULED_OUT": "#7f7f7f"}
bar_colours = [colours.get(t, "#1f77b4") for t in tags]

fig, ax = plt.subplots(figsize=(8, 0.45 * len(codes) + 1.5))
ax.barh(codes, scores, color=bar_colours, edgecolor="k")
ax.set_xlim(0, 1)
ax.set_xlabel("Likelihood score [-]")
ax.set_title("Root-cause candidate ranking")
ax.invert_yaxis()
for i, t in enumerate(tags):
    ax.text(scores[i] + 0.02, i, t, va="center", fontsize=8)
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("rca_candidate_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Save consolidated results

In [ ]:
import json
results = {
    "key_results": {
        "compat_verdict": str(compat.getVerdict()),
        "si_caco3_uninhibited": si_caco3,
        "si_caco3_residual_at_20mgL": residuals[len(residuals)//2],
        "acid_dissolved_kg": float(acid.getScaleDissolvedKg()),
        "acid_co2_generated_kg": float(acid.getCO2GeneratedKg()),
        "scavenger_demand_kg_per_day": float(scav_perf.getH2SRemovedKgPerDay()),
        "scavenger_breakthrough_days": float(scav_perf.getBreakthroughDays()),
        "rca_primary": str(primary.getCode()),
        "rca_primary_score": float(primary.getScore()),
    },
    "figure_captions": {
        "chemical_compatibility_dose_response.png":
            "Residual CaCO3 SI vs. phosphonate inhibitor dose at 80 C",
        "rca_candidate_ranking.png":
            "Ranked root-cause candidates with PRIMARY/CONTRIBUTING/POSSIBLE tags",
    },
}
with open("chemical_compatibility_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))